# Twitch Emote Prediction — Model Training

Trains a fusion model to predict Twitch chat emote distributions from joint audio-visual embeddings. Given a short clip, the model learns which emotes are likely to appear in chat — a proxy for crowd sentiment and audience reaction intensity.

Inputs are produced by the Twitch VOD Scraper (emote targets) and Embedding Extraction (`.npy` vectors) notebooks. Each sample is a concatenated V-JEPA 2 video embedding + Whisper audio embedding paired with a soft emote-frequency target.

- **Model**: `EmoteFusionMLP` — a `CrossAttentionBottleneck` fuses the two modalities, followed by independent residual towers and a shared MLP head. Output is a log-probability distribution over the top-N emotes in the training corpus.
- **Training**: KL-divergence loss against soft emote-frequency targets, with adaptive mixup (α decayed across epochs), per-class temperature scaling, and a cosine-with-warmup learning rate schedule.

### Sections

- **Mounting**: Mounts Google Drive.
- **Unzipping**: Extracts embedding datasets and emote targets from Drive.
- **Imports**: Python and PyTorch dependencies.
- **Datasets**: Configures dataset paths and per-channel identity mappings.
- **Emote Vocabulary**: Tallies emotes across all datasets and builds a global top-N vocabulary.
- **Model Architecture**: `ResidualBlock`, `CrossAttentionBottleneck`, `EmoteFusionMLP`, `TranslatingEmoteDataset`.
- **Dataloaders**: Splits into train/test sets and wraps in DataLoaders.
- **Training**: Training loop with adaptive mixup, temperature warmup, cosine scheduler, and best-model checkpointing.
- **Evaluation**: Training history, per-dataset KL breakdown, quantitative metrics, qualitative inspection, and ranked best predictions.

### Prerequisites

No Colab secrets required. Inputs must be produced by the Twitch VOD Scraper and Embedding Extraction notebooks:

| Path | Contents |
|---|---|
| `dataset_{vod_id}/` | per-clip emote target JSON files and emote vocabulary |
| `embeddings_{vod_id}/` | `.npy` embedding files (one per clip) |

## Mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Unzipping

In [ ]:
import os
import zipfile
from pathlib import Path
from accelerate import Accelerator

# Initialize Accelerator
accelerator = Accelerator()
device = accelerator.device

def log(msg):
    if accelerator.is_main_process:
        print(msg)

# ================= 1. DEFINE YOUR EXTRACTION JOBS =================
# Add or remove dictionaries from this list as needed.
# 'zip_path' is where the file lives. 'extract_to' is where it goes.

ZIP_JOBS = [
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2696878239_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2696878239.zip',
        "extract_to": '/content/embeddings_2696878239'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2697512838_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2697512838.zip',
        "extract_to": '/content/embeddings_2697512838'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2697648768_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2697648768.zip',
        "extract_to": '/content/embeddings_2697648768'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_1594760722_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_1594760722.zip',
        "extract_to": '/content/embeddings_1594760722'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_1541650135_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_1541650135.zip',
        "extract_to": '/content/embeddings_1541650135'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_1755938388_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_1755938388.zip',
        "extract_to": '/content/embeddings_1755938388'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2711972935_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2711972935.zip',
        "extract_to": '/content/embeddings_2711972935'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2711100601_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2711100601.zip',
        "extract_to": '/content/embeddings_2711100601'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2711985062_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2711985062.zip',
        "extract_to": '/content/embeddings_2711985062'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2710135059_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2710135059.zip',
        "extract_to": '/content/embeddings_2710135059'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2713675341_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2713675341.zip',
        "extract_to": '/content/embeddings_2713675341'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/dataset_2712821460_clips.zip',
        "extract_to": '/content'
    },
    {
        "zip_path": '/content/drive/MyDrive/Valorant Group Data/embeddings_2712821460.zip',
        "extract_to": '/content/embeddings_2712821460'
    },
]

# ================= 2. THE EXTRACTION ENGINE =================

for job in ZIP_JOBS:
    zip_file = Path(job["zip_path"])
    extract_dir = Path(job["extract_to"])

    # Check if this is a "clips" zip (extracts to /content) or "embeddings" zip
    # If it's clips, we look for the folder name inside /content
    if "clips.zip" in zip_file.name:
        folder_name = zip_file.name.replace("_clips.zip", "")
        target_check = extract_dir / folder_name
    else:
        target_check = extract_dir

    # Logic: Skip if the target folder exists and is not empty
    if target_check.exists() and any(target_check.iterdir()):
        log(f"Skipping '{zip_file.name}' - already extracted to {target_check.name}")
        continue

    if zip_file.exists():
        log(f"Unzipping '{zip_file.name}' to {extract_dir}...")

        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        log(f"Done unzipping {zip_file.name}.")
        log("-" * 40)
    else:
        log(f"❌ Zip file not found: {zip_file}")
        log("-" * 40)

# Print hardware status once at the end
log(f"Running on {accelerator.num_processes} GPU(s) | Device: {device}")

## Imports

In [ ]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from pathlib import Path
from collections import defaultdict
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter


## Datasets

In [ ]:
# 0 = main valorant, 1 = FNS, 2 = VCT Americas, 3 = VCT EMEA
DATASETS = [
    {'vod_id': '1541650135', 'channel': 0},
    {'vod_id': '1594760722', 'channel': 0},
    {'vod_id': '1755938388', 'channel': 0},
    # {'vod_id': '2696878239', 'channel': 3},
    # {'vod_id': '2697512838', 'channel': 2},
    # {'vod_id': '2697648768', 'channel': 1},
    # {'vod_id': '2711100601', 'channel': 1},
    # {'vod_id': '2711972935', 'channel': 1},
    # {'vod_id': '2710135059', 'channel': 1},
    # {'vod_id': '2711985062', 'channel': 1},
    # {'vod_id': '2713675341', 'channel': 1},
    # {'vod_id': '2712821460', 'channel': 1},
]

DATASET_PATHS = [{
    'base':       f'/content/dataset_{d["vod_id"]}',
    'target':     f'/content/dataset_{d["vod_id"]}/target',
    'mapping':    f'/content/dataset_{d["vod_id"]}/emote_mapping.json',
    'embed':      f'/content/embeddings_{d["vod_id"]}',
    'channel_id': d['channel'],
} for d in DATASETS]

log(f"Active datasets: {len(DATASET_PATHS)}.")

## Emote Vocabulary

In [ ]:
TOP_N = 50  # Only keep the most common emotes

log("Tallying emotes across all datasets...")
global_emote_counts = defaultdict(float)

for paths in DATASET_PATHS:
    # 1. Load the local dictionary {"EmoteName": LocalIndex}
    with open(paths['mapping'], 'r') as f:
        local_map = json.load(f)

    # Reverse it to {LocalIndex: "EmoteName"} so we can look up names by index
    idx_to_name = {v: k for k, v in local_map.items()}

    # 2. Scan all target files in this dataset
    target_files = [f for f in os.listdir(paths['target']) if f.endswith('.json')]
    for fname in target_files:
        with open(os.path.join(paths['target'], fname), 'r') as f:
            target_array = json.load(f)

            # 3. Add the clip's scores to the Global Dictionary
            for idx, score in enumerate(target_array):
                if score > 0:  # Only count emotes that actually appeared
                    emote_name = idx_to_name.get(idx)
                    if emote_name:
                        global_emote_counts[emote_name] += score

# 4. Sort and keep only the top-N most common emotes
sorted_emotes = sorted(global_emote_counts.items(), key=lambda x: x[1], reverse=True)
all_emote_names = [name for name, count in sorted_emotes[:TOP_N]]

# 5. Create our UNIVERSAL DICTIONARY {"EmoteName": GlobalIndex}
global_vocab = {name: i for i, name in enumerate(all_emote_names)}

total_emotes = len(global_vocab)
log(f"Created global vocabulary with top {total_emotes} emotes (out of {len(sorted_emotes)} total).")
log(f"Top 5 emotes: {all_emote_names[:5]}")
log(f"Bottom 5 of kept emotes: {all_emote_names[-5:]}")

## Model Architecture

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.7):
        super().__init__()
        self.ln = nn.LayerNorm(dim)

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(self.ln(x))


class CrossAttentionBottleneck(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        # batch_first=True means inputs should be [Batch, Seq_Len, Dim]
        self.v_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.a_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True)

        # LayerNorms for the residual connections
        self.norm_v = nn.LayerNorm(dim)
        self.norm_a = nn.LayerNorm(dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, visual, audio):
        # 1. Reshape from [Batch, 128] -> [Batch, 1, 128] to act as a "sequence"
        v_seq = visual.unsqueeze(1)
        a_seq = audio.unsqueeze(1)

        # 2. Visual queries Audio: "Do you have any audio details that match my visual scene?"
        # Query = Visual | Key = Audio | Value = Audio
        v_attn_out, _ = self.v_attn(query=v_seq, key=a_seq, value=a_seq)

        # 3. Audio queries Visual: "Do you have any visual events that match my audio spike?"
        # Query = Audio | Key = Visual | Value = Visual
        a_attn_out, _ = self.a_attn(query=a_seq, key=v_seq, value=v_seq)

        # 4. Squeeze back down to [Batch, 128]
        v_attn_out = v_attn_out.squeeze(1)
        a_attn_out = a_attn_out.squeeze(1)

        # 5. Add the attended features back to the original features (Residual + Norm)
        v_out = self.norm_v(visual + self.dropout(v_attn_out))
        a_out = self.norm_a(audio + self.dropout(a_attn_out))

        return v_out, a_out


class EmoteFusionMLP(nn.Module):
    def __init__(self, visual_dim=1024, audio_dim=768, hidden_dim=512, output_dim=300, num_channels=4, dropout=0.7):
        super().__init__()
        self.channel_dim = 64

        # Constants for the funnel
        tower_dim = 64
        tower_dropout = 0.2

        # 3-Step Gradual Bottleneck: Visual
        self.visual_tower = nn.Sequential(
            nn.LayerNorm(visual_dim),
            nn.Linear(visual_dim, 256),
            nn.GELU(),
            nn.Dropout(tower_dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, tower_dim),
            nn.GELU(),
        )

        # 3-Step Gradual Bottleneck: Audio
        self.audio_tower = nn.Sequential(
            nn.LayerNorm(audio_dim),
            nn.Linear(audio_dim, 256),
            nn.GELU(),
            nn.Dropout(tower_dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, tower_dim),
            nn.GELU(),
        )

        self.cross_modal = CrossAttentionBottleneck(dim=tower_dim, num_heads=8, dropout=0.1)

        self.channel_embed = nn.Embedding(num_channels, self.channel_dim)

        self.fusion_projection = nn.Sequential(
            nn.Linear(tower_dim * 2 + self.channel_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )

        self.blocks = nn.Sequential(
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
        )

        self.final_norm = nn.LayerNorm(hidden_dim)

        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim, output_dim))

        # Per-class temperature scaling
        self.temperature = nn.Parameter(torch.ones(output_dim))

    def forward(self, x, channel_id):
        if self.training:
            x = F.dropout(x, p=0.1)

        visual = self.visual_tower(x[:, :1024])
        audio  = self.audio_tower(x[:, 1024:])

        visual, audio = self.cross_modal(visual, audio)

        fused = torch.cat([visual, audio, self.channel_embed(channel_id)], dim=-1)
        fused = self.fusion_projection(fused)

        fused = self.blocks(fused)
        fused = self.final_norm(fused)

        temp = self.temperature.unsqueeze(0).clamp(min=0.5, max=3.0)
        logits = self.head(fused) / temp
        return F.log_softmax(logits, dim=-1)

class TranslatingEmoteDataset(Dataset):
    def __init__(self, embed_dir, target_dir, local_mapping_file, global_vocab, channel_id):
        self.embed_dir  = Path(embed_dir)
        self.target_dir = Path(target_dir)
        self.global_vocab = global_vocab
        self.channel_id = channel_id

        with open(local_mapping_file, 'r') as f:
            self.local_map = json.load(f)

        valid_local_indices = {
            local_idx
            for name, local_idx in self.local_map.items()
            if name in self.global_vocab
        }

        all_files = [f for f in os.listdir(target_dir) if f.endswith('.json')]
        self.target_files = []

        print(f"Scanning {len(all_files)} clips in {self.target_dir.name} (channel {channel_id})...")
        for f_name in all_files:
            with open(self.target_dir / f_name, 'r') as f:
                local_target = json.load(f)

            is_valid = any(
                local_target[idx] > 0
                for idx in valid_local_indices
                if idx < len(local_target)
            )
            if is_valid:
                self.target_files.append(f_name)

        dropped = len(all_files) - len(self.target_files)
        print(f"Kept {len(self.target_files)} clips, dropped {dropped}.")

    def __len__(self):
        return len(self.target_files)

    def __getitem__(self, idx):
        target_fname = self.target_files[idx]
        embed_fname  = f"{Path(target_fname).stem}_embeddings.npy"

        embedding = np.load(self.embed_dir / embed_fname)

        with open(self.target_dir / target_fname, 'r') as f:
            local_target = np.array(json.load(f), dtype=np.float32)

        global_target = np.zeros(len(self.global_vocab), dtype=np.float32)

        # Track which indices are actually valid for this channel
        valid_indices = []

        for name, global_idx in self.global_vocab.items():
            if name in self.local_map:
                valid_indices.append(global_idx) # Keep track of "legal" emotes
                l_idx = self.local_map[name]
                if l_idx < len(local_target):
                    global_target[global_idx] = local_target[l_idx]

        target_sum = global_target.sum()
        if target_sum == 0:
            raise ValueError(f"CRITICAL: Clip {target_fname} has 0 valid emotes but slipped through __init__ filtering!")

        # Normalize the raw counts first
        global_target /= target_sum

        # This prevents the model from ever "guessing" an emote that isn't on the channel
        label_smoothing = 0.025 # Reserve 2.5% of the probability mass for uncertainty
        num_valid = len(valid_indices)

        for idx in valid_indices:
            # Shift the probability slightly towards uniform
            global_target[idx] = global_target[idx] * (1.0 - label_smoothing) + (label_smoothing / num_valid)

        # Re-normalize just to be safe
        global_target /= global_target.sum()

        clip_path = str(self.embed_dir / embed_fname)
        return (
            torch.from_numpy(embedding).float(),
            torch.from_numpy(global_target).float(),
            torch.tensor(self.channel_id, dtype=torch.long),
            clip_path
        )

## Dataloaders

In [ ]:
individual_datasets = []
for paths in DATASET_PATHS:
    ds = TranslatingEmoteDataset(
        embed_dir=paths['embed'],
        target_dir=paths['target'],
        local_mapping_file=paths['mapping'],
        global_vocab=global_vocab,
        channel_id=paths['channel_id'],
    )
    individual_datasets.append(ds)

dataset = ConcatDataset(individual_datasets)
log(f"Total clips: {len(dataset)}")

train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

log("-" * 30)
log(f"Training samples: {len(train_dataset)}")
log(f"Test samples:     {len(test_dataset)}")
log("-" * 30)
log(f"Train batches: {len(train_loader)}")
log(f"Test batches:  {len(test_loader)}")

## Training

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log(f"Training on: {device}")
num_channels = max(p['channel_id'] for p in DATASET_PATHS) + 1
model = EmoteFusionMLP(visual_dim=1024, audio_dim=768, hidden_dim=64, output_dim=len(global_vocab), num_channels=num_channels).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"Trainable parameters: {total_params:,}")

criterion = nn.KLDivLoss(reduction='batchmean')
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

epochs = 100
patience = 12
WARMUP_EPOCHS = 5

# Linear warmup from 10% of lr up to full lr, then cosine anneal for the remainder
warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS
)

cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs - WARMUP_EPOCHS, eta_min=1e-6
)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS]
)

history = {'train_loss': [], 'test_loss': []}
best_v_loss = float('inf')
epochs_without_improvement = 0

log("Starting training loop...")
for epoch in range(epochs):

    # Freeze temperature scaling while the model learns basic features
    if epoch < WARMUP_EPOCHS:
        model.temperature.requires_grad = False
    else:
        model.temperature.requires_grad = True

    if epoch < 20:
        current_alpha = 0.5  # Strong mixup early on
    elif epoch < 35:
        current_alpha = 0.2  # Weaker mixup as it settles
    else:
        current_alpha = 0.0  # Pure data for final fine-tuning

    # --- TRAINING ---
    model.train()
    total_train_loss = 0

    for embeds, targets, channel_ids, _ in train_loader:
        embeds, targets, channel_ids = embeds.to(device), targets.to(device), channel_ids.to(device)

        # Feature level noise
        embeds = embeds + torch.randn_like(embeds) * 0.02

        if current_alpha > 0:
            lam = np.random.beta(current_alpha, current_alpha)
            idx = torch.randperm(embeds.size(0), device=device)
            same_channel = (channel_ids == channel_ids[idx]).float().unsqueeze(1)

            embeds  = lam * embeds  + (1 - lam) * (same_channel * embeds[idx]  + (1 - same_channel) * embeds)
            targets = lam * targets + (1 - lam) * (same_channel * targets[idx] + (1 - same_channel) * targets)

        optimizer.zero_grad()
        log_preds = model(embeds, channel_ids)
        loss = criterion(log_preds, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item()

    # --- VALIDATION (no mixup, no noise) ---
    model.eval()
    total_test_loss = 0
    with torch.no_grad():
        for embeds, targets, channel_ids, _ in test_loader:
            embeds, targets, channel_ids = embeds.to(device), targets.to(device), channel_ids.to(device)
            log_preds = model(embeds, channel_ids)
            loss = criterion(log_preds, targets)
            total_test_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)
    avg_test  = total_test_loss  / len(test_loader)

    history['train_loss'].append(avg_train)
    history['test_loss'].append(avg_test)
    scheduler.step()

    if avg_test < best_v_loss:
        best_v_loss = avg_test
        torch.save(model.state_dict(), 'best_model.pth')
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    current_lr  = optimizer.param_groups[0]['lr']
    temp_min = model.temperature.min().item()
    temp_max = model.temperature.max().item()

    phase = "warmup" if epoch < WARMUP_EPOCHS else "cosine"

    log(f"Epoch {epoch+1:02d}/{epochs} [{phase}] | Mixup: {current_alpha:.1f} | Train: {avg_train:.4f} | Test: {avg_test:.4f} | LR: {current_lr:.6f} | Temp Range: {temp_min:.2f} to {temp_max:.2f}")

    if epochs_without_improvement >= patience:
        log(f"\nEarly stop after {patience} epochs without improvement.")
        log(f"Best model saved with test loss: {best_v_loss:.4f}")
        break

## Export

In [ ]:
import shutil

# ← Set your Drive export path here
EXPORT_PATH = '/content/drive/MyDrive/twitch_emote_data/best_model.pth'

shutil.copy('/content/best_model.pth', EXPORT_PATH)
log(f"Saved best_model.pth to {EXPORT_PATH}")

## Evaluation

### Training History

In [ ]:
def plot_training_history(history):
    epochs = range(1, len(history['train_loss']) + 1)

    plt.style.use('seaborn-v0_8-muted')
    plt.figure(figsize=(10, 6))

    plt.plot(epochs, history['train_loss'], 'o-', label='Train KL-Loss', markersize=4, linewidth=2)
    plt.plot(epochs, history['test_loss'], 's-', label='Test KL-Loss', markersize=4, linewidth=2)

    best_epoch = history['test_loss'].index(min(history['test_loss'])) + 1
    plt.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Model (Epoch {best_epoch})')

    plt.title('EmoteFusionMLP: Training vs. Test KL-Divergence', fontsize=14, pad=15)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('KL-Loss (Batch Mean)', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(frameon=True, facecolor='white', framealpha=0.9)

    plt.tight_layout()

    plt.savefig('training_loss_visualization.png', dpi=300)
    print(f"Saved 'training_loss_visualization.png' — best test loss: {min(history['test_loss']):.4f}")

plot_training_history(history)

### Per-Dataset Breakdown

In [ ]:
train_indices_set = set(train_dataset.indices)

model_path = '/content/best_model.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))
    print(f"Loaded '{model_path}'")
else:
    print("⚠️ Could not find the file. Testing the current model instead.")

model.eval()
print(f"\n{'Dataset':<15} {'Ch':<4} {'Train KL':<12} {'Test KL':<12} {'Train N':<10} {'Test N'}")
print("-" * 65)

with torch.no_grad():
    offset = 0
    for ds, paths in zip(individual_datasets, DATASET_PATHS):
        train_loss, test_loss = 0.0, 0.0
        train_n, test_n = 0, 0

        for local_i in range(len(ds)):
            global_i = local_i + offset
            embed, target, channel_id, _ = ds[local_i]
            embed      = embed.unsqueeze(0).to(device)
            target     = target.unsqueeze(0).to(device)
            channel_id = channel_id.unsqueeze(0).to(device)

            log_pred = model(embed, channel_id)
            loss = F.kl_div(log_pred, target, reduction='batchmean').item()

            if global_i in train_indices_set:
                train_loss += loss; train_n += 1
            else:
                test_loss  += loss; test_n  += 1

        folder_id = paths['base'].split('_')[-1]
        tr = train_loss/train_n if train_n else float('nan')
        te = test_loss/test_n   if test_n  else float('nan')
        print(f"{folder_id:<15} {paths['channel_id']:<4} {tr:<12.4f} {te:<12.4f} {train_n:<10} {test_n}")
        offset += len(ds)

### Quantitative Metrics

In [ ]:
# --- 1. Setup ---
model_path = '/content/best_model.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))
    print(f"Loaded '{model_path}'")
else:
    print("⚠️ Could not find the file. Testing the current model instead.")
model.eval()
device = next(model.parameters()).device

# --- 2. Configuration ---
NUM_TESTS = 1623
TOP_K = 5

kl_losses = []
overlap_results = []
top1_in_top5_hits = 0
total_samples = 0

print(f"Running evaluation on {NUM_TESTS} clips...\n")

# --- 3. Evaluation Loop ---
with torch.no_grad():
    for embeds, targets, channel_ids, _ in test_loader:
        if total_samples >= NUM_TESTS:
            break

        embeds, targets, channel_ids = embeds.to(device), targets.to(device), channel_ids.to(device)
        log_preds = model(embeds, channel_ids)

        pred_probs   = torch.exp(log_preds).cpu().numpy()
        target_probs = targets.cpu().numpy()

        for i in range(embeds.size(0)):
            if total_samples >= NUM_TESTS:
                break

            sample_loss = F.kl_div(log_preds[i:i+1], targets[i:i+1], reduction='batchmean').item()
            kl_losses.append(sample_loss)

            top_actual = set(target_probs[i].argsort()[-TOP_K:][::-1])
            top_pred   = pred_probs[i].argsort()[-TOP_K:][::-1]

            # Metric A: Top-1 in Top-K accuracy
            if top_pred[0] in top_actual:
                top1_in_top5_hits += 1

            # Metric B: Raw overlap count (0 to TOP_K)
            overlap = sum(1 for p in top_pred if p in top_actual)
            overlap_results.append(overlap)
            total_samples += 1

# --- 4. Report ---
avg_kl      = np.mean(kl_losses)
top1_acc    = (top1_in_top5_hits / total_samples) * 100
avg_overlap = np.mean(overlap_results)

print("-" * 50)
print(f"Evaluation report ({total_samples} clips)")
print("-" * 50)
print(f"Average KL loss:      {avg_kl:.4f}")
print(f"Top-1 in top-5 acc:   {top1_acc:.1f}%")
print(f"Top-5 overlap avg:    {avg_overlap:.2f} / {TOP_K} ({(avg_overlap / TOP_K) * 100:.1f}%)")
print("-" * 50)

# --- 5. Overlap Distribution ---
counts = Counter(overlap_results)
dist = [counts.get(i, 0) for i in range(TOP_K + 1)]
pcts  = [(c / total_samples) * 100 for c in dist]

plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.8, TOP_K + 1))
bars = plt.bar(range(TOP_K + 1), dist, color=colors, edgecolor='black', alpha=0.8)

for bar, pct in zip(bars, pcts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (max(dist) * 0.01),
             f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.title(f'Top-{TOP_K} Emote Overlap Distribution', fontsize=14)
plt.xlabel('Number of Matching Emotes (Predicted vs Actual)', fontsize=12)
plt.ylabel('Number of Clips', fontsize=12)
plt.xticks(range(TOP_K + 1))
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

# --- 6. KL-Loss Distribution ---
plt.figure(figsize=(12, 6))
sns.histplot(kl_losses, kde=True, color='royalblue', bins=50, alpha=0.4, line_kws={'lw': 3})
plt.axvline(avg_kl, color='red', linestyle='--', label=f'Mean KL: {avg_kl:.4f}')

plt.title(f'KL-Loss Distribution ({total_samples} Clips)', fontsize=15, pad=15)
plt.xlabel('KL-Divergence (per clip)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

stats_text = (f"Min: {min(kl_losses):.3f}\n"
              f"Max: {max(kl_losses):.3f}\n"
              f"Std: {np.std(kl_losses):.3f}\n"
              f"Avg overlap: {avg_overlap:.2f}/{TOP_K}")
plt.gca().text(0.95, 0.70, stats_text, transform=plt.gca().transAxes,
               fontsize=10, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('kl_loss_distribution.png', dpi=300)
plt.show()

print(f"Saved 'kl_loss_distribution.png'")

### Qualitative Inspection

In [ ]:
model_path = '/content/best_model.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))
    print(f"Loaded '{model_path}'")
else:
    print("⚠️ Could not find the file. Testing the current model instead.")

model.eval()
check_ds = test_dataset
device = next(model.parameters()).device
idx_to_name = {v: k for k, v in check_ds.dataset.datasets[0].global_vocab.items()}

SHOW_TOP_N = 10
print(f"Sample predictions (random) — top {SHOW_TOP_N} emotes per clip:\n")

with torch.no_grad():
    for _ in range(5):
        idx = random.randint(0, len(check_ds) - 1)

        embed, target, channel_id, clip_path = check_ds[idx]

        log_pred = model(embed.unsqueeze(0).to(device), channel_id.unsqueeze(0).to(device))

        target_tensor = target.unsqueeze(0).to(device)
        kl_score = F.kl_div(log_pred, target_tensor, reduction='batchmean').item()

        pred_probs   = torch.exp(log_pred).squeeze(0).cpu().numpy()
        target_probs = target.numpy()

        top_actual_indices = target_probs.argsort()[-SHOW_TOP_N:][::-1]
        top_pred_indices   = pred_probs.argsort()[-SHOW_TOP_N:][::-1]

        top_guess_name = idx_to_name[top_pred_indices[0]]
        is_hit = any(idx_to_name[i] == top_guess_name for i in top_actual_indices if target_probs[i] > 0)
        status = "match" if is_hit else "❌ miss"

        actual_list = [f"{idx_to_name[i]} ({target_probs[i]:.2f})" for i in top_actual_indices if target_probs[i] > 0.01]
        pred_list   = [f"{idx_to_name[i]} ({pred_probs[i]:.2f})" for i in top_pred_indices]

        print(f"Clip: {os.path.basename(clip_path)} | {status} | KL: {kl_score:.4f} | channel: {channel_id.item()}")
        print(f"  Actual (top {SHOW_TOP_N}):    {', '.join(actual_list) if actual_list else 'No dominant emotes'}")
        print(f"  Predicted (top {SHOW_TOP_N}): {', '.join(pred_list)}")
        print("-" * 70)

### Best Predictions

In [ ]:
# --- 1. Configuration ---
NUM_TO_SCAN = 1623
NUM_TO_SHOW = 20
SHOW_TOP_N_EMOTES = 5

# --- 2. Setup ---
model_path = '/content/best_model.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))
    print(f"Loaded '{model_path}'")

model.eval()
check_ds = test_dataset
device = next(model.parameters()).device
idx_to_name = {v: k for k, v in check_ds.dataset.datasets[0].global_vocab.items()}

results = []
print(f"Scanning {NUM_TO_SCAN} clips for top {NUM_TO_SHOW} best predictions...\n")

# --- 3. Scan ---
with torch.no_grad():
    indices = random.sample(range(len(check_ds)), min(NUM_TO_SCAN, len(check_ds)))

    for idx in indices:
        embed, target, channel_id, clip_path = check_ds[idx]

        log_pred = model(embed.unsqueeze(0).to(device), channel_id.unsqueeze(0).to(device))
        target_tensor = target.unsqueeze(0).to(device)

        kl_score = F.kl_div(log_pred, target_tensor, reduction='batchmean').item()

        results.append({
            'kl_score':   kl_score,
            'clip_name':  os.path.basename(clip_path),
            'channel':    channel_id.item(),
            'pred_probs': torch.exp(log_pred).squeeze(0).cpu().numpy(),
            'target_probs': target.numpy()
        })

# --- 4. Sort and Display ---
# Sort by KL-Loss (lowest = best prediction)
results.sort(key=lambda x: x['kl_score'])

print(f"Top {NUM_TO_SHOW} best predictions:\n")
for i in range(min(NUM_TO_SHOW, len(results))):
    res = results[i]

    top_actual_idx = res['target_probs'].argsort()[-SHOW_TOP_N_EMOTES:][::-1]
    top_pred_idx   = res['pred_probs'].argsort()[-SHOW_TOP_N_EMOTES:][::-1]

    print(f"#{i+1} | {res['clip_name']} | KL: {res['kl_score']:.4f} | channel: {res['channel']}")

    actual_str = ", ".join([f"{idx_to_name[idx]} ({res['target_probs'][idx]:.2f})" for idx in top_actual_idx if res['target_probs'][idx] > 0.01])
    pred_str   = ", ".join([f"{idx_to_name[idx]} ({res['pred_probs'][idx]:.2f})" for idx in top_pred_idx])

    print(f"  Actual:    {actual_str}")
    print(f"  Predicted: {pred_str}")
    print("-" * 80)